# Laminated composite cantilever: tension and bending of an orthotropic shell

This example models a rectangular five-ply laminate using the same displacement–rotation weak-form formulation as the U-channel shell tutorial. The plies have different elastic moduli along and across the fibers. Their orientations and thickness coordinates determine the membrane, membrane–bending coupling, bending and transverse-shear stiffnesses.

The example uses a structured quadrilateral mesh, the parameter-integration overload of `∫` for through-thickness integration, and distributed edge loads. No multipoint constraints or dedicated composite shell element are needed. All dimensions are in mm, forces in N and stresses in N/mm².

In [1]:
using LowLevelFEM, LinearAlgebra

## 1. Geometry and unknown fields

Create a rectangular midsurface in the global $xy$ plane, 200 mm long and 80 mm wide. The five plies share this mesh, so no separate through-thickness mesh is required. The edge `left` is clamped; the edge `right` carries a tensile force and an applied bending moment.

The displacement $\mathbf u$ and infinitesimal rotation $\boldsymbol\varphi$ each have three global components. The `Material` object associates the field descriptors with `body`; the orthotropic stiffness will be specified explicitly rather than taken from the material's default isotropic law.

In [2]:
Lx, Ly = 200.0, 80.0
nx, ny = 40, 16
structured_rect_mesh(lx=Lx, ly=Ly, nx=nx, ny=ny, order=1)

surf = "body"
mat = Material(surf)
U = Field([mat]; type=:VectorField, dim=3, fieldName=:u, rhsName=:f)
Φ = Field([mat]; type=:VectorField, dim=3, fieldName=:φ, rhsName=:m);

## 2. Orthotropic lamina constitutive law

In the material principal directions (1 along the fibers, 2 across the fibers), choose representative linear-elastic moduli $E_1=135$ GPa, $E_2=10$ GPa, $G_{12}=5$ GPa, $G_{13}=5$ GPa, $G_{23}=3.8$ GPa, and $\nu_{12}=0.3$. The reciprocal Poisson ratio is $\nu_{21}=\nu_{12}E_2/E_1$.

For plane stress and engineering shear strain, the reduced lamina stiffness is

$$
\mathbf Q_0 = \begin{bmatrix}
E_1/\Delta & \nu_{12}E_2/\Delta & 0\\
\nu_{12}E_2/\Delta & E_2/\Delta & 0\\
0 & 0 & G_{12}
\end{bmatrix},\qquad \Delta=1-\nu_{12}\nu_{21}.
$$

For a ply at angle $\theta$ measured from the global $x$ axis toward $y$, let $c=\cos\theta$, $s=\sin\theta$. Its engineering strain transformation from global to material coordinates is

$$
\mathbf T_\varepsilon(\theta)=\begin{bmatrix}
c^2&s^2&cs\\s^2&c^2&-cs\\-2cs&2cs&c^2-s^2
\end{bmatrix}.
$$

Preservation of virtual work gives the global reduced stiffness $\overline{\mathbf Q}=\mathbf T_\varepsilon^\mathsf T\mathbf Q_0\mathbf T_\varepsilon$. The transverse-shear stiffness is rotated analogously, taking care to use the shell's shear ordering $[\gamma_{yz},\gamma_{xz}]^\mathsf T$.

More explicitly, for the chosen transverse-shear ordering,

$$
\mathbf T_s(\theta)=\begin{bmatrix}s&c\\c&-s\end{bmatrix},\qquad
\overline{\mathbf G}=\mathbf T_s^\mathsf T\operatorname{diag}(G_{13},G_{23})\mathbf T_s.
$$

In [3]:
E1, E2 = 135e3, 10e3
ν12 = 0.3
ν21 = ν12 * E2 / E1
G12, G13, G23 = 5e3, 5e3, 3.8e3
Δ = 1 - ν12*ν21

Q0 = [E1/Δ    ν12*E2/Δ  0
      ν12*E2/Δ E2/Δ     0
      0        0        G12]

function Tε(θ)
    c, s = cosd(θ), sind(θ)
    [c^2   s^2   c*s
     s^2   c^2  -c*s
    -2c*s  2c*s  c^2-s^2]
end

Qbar(θ) = Tε(θ)' * Q0 * Tε(θ)

function Gbar(θ)
    c, s = cosd(θ), sind(θ)
    Ts = [s c; c -s]
    Ts' * Diagonal([G13, G23]) * Ts
end;

## 3. Through-thickness integration of five plies

The layup, from the negative to the positive thickness coordinate, is $[0^\circ,45^\circ,90^\circ,-45^\circ,0^\circ]$. Each ply is 0.5 mm thick. The $0^\circ$ plies are stiff along $x$, the $90^\circ$ ply along $y$, and the $\pm45^\circ$ plies contribute oblique reinforcement. This is a balanced in-plane layup but not a symmetric layup, so the membrane–bending coupling matrix need not vanish.

For ply interfaces $z_k$, directly integrate the laminate stiffness moments:

$$
\mathbf A=\sum_k\int_{z_k}^{z_{k+1}}\overline{\mathbf Q}_k\,\mathrm d\zeta,\qquad
\mathbf B=\sum_k\int_{z_k}^{z_{k+1}}\zeta\overline{\mathbf Q}_k\,\mathrm d\zeta,\qquad
\mathbf D=\sum_k\int_{z_k}^{z_{k+1}}\zeta^2\overline{\mathbf Q}_k\,\mathrm d\zeta.
$$

The generalized membrane–bending and transverse-shear stiffnesses are

$$
\mathbf D_{mb}=\begin{bmatrix}\mathbf A&\mathbf B\\\mathbf B&\mathbf D\end{bmatrix},\qquad
\mathbf D_s=k_s\sum_k\int_{z_k}^{z_{k+1}}\overline{\mathbf G}_k\,\mathrm d\zeta.
$$

The parameter integral `∫(f, a, b)` makes the code closely follow these formulas. Here $k_s=5/6$ is a common illustrative shear correction; it is not a layer-specific through-thickness shear solution.

In [4]:
θ = [0.0, 45.0, 90.0, -45.0, 0.0]
tply = 0.5
h = length(θ) * tply
z = collect(range(-h/2, h/2; length=length(θ)+1))
Qlay = Qbar.(θ)
Glay = Gbar.(θ)
k_s = 5/6

A_lam = sum(∫(ζ -> Qlay[k],     z[k], z[k+1]) for k in eachindex(θ))
B_lam = sum(∫(ζ -> ζ*Qlay[k],   z[k], z[k+1]) for k in eachindex(θ))
D_lam = sum(∫(ζ -> ζ^2*Qlay[k], z[k], z[k+1]) for k in eachindex(θ))
D_mb = [A_lam B_lam;
        B_lam D_lam]

D_s = k_s * sum(∫(ζ -> Glay[k], z[k], z[k+1]) for k in eachindex(θ))
D_mb

6×6 Matrix{Float64}:
 183943.0           37533.6          …       3.63798e-12  -15729.9
  37533.6               1.21023e5            6.36646e-12  -15729.9
      0.0               0.0             -15729.9              -4.54747e-13
      0.0               3.63798e-12      12052.9              -9.09495e-13
      3.63798e-12       6.36646e-12      23339.3              -9.09495e-13
 -15729.9          -15729.9          …      -1.81899e-12   14630.9

## 4. Shell kinematics and weak form

For this flat midsurface, the local frame is simply $\mathbf t_\xi=\mathbf e_x$, $\mathbf t_\eta=\mathbf e_y$, $\mathbf n=\mathbf e_z$. With the physical rotation vector, the through-thickness displacement is

$$
\mathbf u(\xi,\eta,\zeta)=\mathbf u(\xi,\eta)+\zeta\,\boldsymbol\varphi\times\mathbf n.
$$

Thus the in-plane strain is $\boldsymbol\varepsilon(\zeta)=\boldsymbol\varepsilon_m+\zeta\boldsymbol\kappa_b$. In the fixed Cartesian frame the strain components are

$$
\boldsymbol\varepsilon_m=\begin{bmatrix}u_{x,x}\\u_{y,y}\\u_{x,y}+u_{y,x}\end{bmatrix},\qquad
\boldsymbol\kappa_b=\begin{bmatrix}\varphi_{y,x}\\-\varphi_{x,y}\\\varphi_{y,y}-\varphi_{x,x}\end{bmatrix},
$$

$$
\gamma_\eta=u_{z,y}-\varphi_x,\qquad\gamma_\xi=u_{z,x}+\varphi_y.
$$

As in the U-channel tutorial, the generalized strain order is $[\boldsymbol\varepsilon_m^\mathsf T,\boldsymbol\kappa_b^\mathsf T,\gamma_\eta,\gamma_\xi]^\mathsf T$. The following operator matrices express these kinematics using `Grad` and the independent rotation field. The complete membrane–bending term retains the off-diagonal laminate matrix $\mathbf B$.

In [5]:
tξ = [1.0, 0.0, 0.0]
tη = [0.0, 1.0, 0.0]
n  = [0.0, 0.0, 1.0]

A_u = vcat(kron(tξ,tξ)',
           kron(tη,tη)',
           (kron(tξ,tη) + kron(tη,tξ))',
           zeros(3,9),
           kron(n,tη)',
           kron(n,tξ)')

A_φ = vcat(zeros(3,9),
           kron(tη,tξ)',
           -kron(tξ,tη)',
           (kron(tη,tη) - kron(tξ,tξ))',
           zeros(2,9))

G_φ = vcat(zeros(6,3), -tξ', tη');

The internal virtual work is

$$
\delta W_{\mathrm{int}}=\int_{\Gamma_s}
\begin{bmatrix}\delta\boldsymbol\varepsilon_m\\\delta\boldsymbol\kappa_b\end{bmatrix}^{\mathsf T}
\mathbf D_{mb}
\begin{bmatrix}\boldsymbol\varepsilon_m\\\boldsymbol\kappa_b\end{bmatrix}\,\mathrm d\Gamma
+\int_{\Gamma_s}\delta\boldsymbol\gamma^{\mathsf T}\mathbf D_s\boldsymbol\gamma\,\mathrm d\Gamma.
$$

Use full quadrature for the membrane and bending contribution, and reduced quadrature for transverse shear to alleviate shear locking. As in the U-channel example, add a small drilling penalty based on $hG_{12}$ and $\alpha=10^{-4}$; this ties the normal rotation to the midsurface rotation via the penalty strain $\gamma_d=\varphi_z+(u_{x,y}-u_{y,x})/2$, rather than introducing a physical drilling stiffness.

In [6]:
B_mb = A_u[1:6, :] ⋅ Grad(U) + A_φ[1:6, :] ⋅ Grad(Φ)
B_s  = A_u[7:8, :] ⋅ Grad(U) + G_φ[7:8, :] ⋅ Φ

K = ∫(B_mb' ⋅ D_mb ⋅ B_mb; Γ=surf) +
    ∫(B_s' ⋅ D_s ⋅ B_s; Γ=surf, gauss=:reduced)

A_d = (kron(tξ,tη) - kron(tη,tξ))' / 2
B_d = A_d ⋅ Grad(U) + n' ⋅ Φ
α = 1e-4
Kd = ∫(B_d' ⋅ (α*h*G12) ⋅ B_d; Γ=surf);

## 5. Combined tension and bending

Clamp both fields along `left`. Apply a total tensile force $F_x=2000$ N and a total bending moment $M_y=600$ N·mm, distributed uniformly over the right edge. Accordingly, the displacement field receives the line force $F_x/L_y$, and the rotation field receives the line moment $M_y/L_y$ about the global $y$ axis. Neither load requires a remote point or an MPC.

In [7]:
Fx, My = 2000.0, 600.0
fu = ∫(U ⋅ [Fx/Ly, 0.0, 0.0]; Γ="right")
fφ = ∫(Φ ⋅ [0.0, My/Ly, 0.0]; Γ="right")
F = SystemVector([fu, fφ])

bc_u = BoundaryCondition("left"; field=U, ux=0, uy=0, uz=0)
bc_φ = BoundaryCondition("left"; field=Φ, φx=0, φy=0, φz=0)
u, φ = solveField(Symmetric(K + Kd), F; support=[bc_u, bc_φ]);

In [8]:
showDoFResults(u; name="Displacement", visible=true, factor=5);

## 6. Recover laminate stress resultants

For postprocessing, represent the Cartesian shell basis by constant vector fields and evaluate the solved field gradients on the midsurface. The recovered membrane strains, curvatures and shear strains follow the same sign and component conventions as the assembly.

The constitutive relation is

$$
\begin{bmatrix}\mathbf N\\\mathbf M\end{bmatrix}=\mathbf D_{mb}
\begin{bmatrix}\boldsymbol\varepsilon_m\\\boldsymbol\kappa_b\end{bmatrix},\qquad
\begin{bmatrix}Q_\eta\\Q_\xi\end{bmatrix}=\mathbf D_s\begin{bmatrix}\gamma_\eta\\\gamma_\xi\end{bmatrix}.
$$

The shear-strain element-center values are recovered by the nodal mean for this affine Q1 mesh, consistently with the reduced shear quadrature.

In [9]:
tξ = VectorField(U, surf, [1.0, 0.0, 0.0])
tη = VectorField(U, surf, [0.0, 1.0, 0.0])
n  = VectorField(U, surf, [0.0, 0.0, 1.0])
φe = nodesToElements(φ; onPhysicalGroup=surf)

gu = ∇(u)
gφ = ∇(φe)
uξ, uη = gu*tξ, gu*tη
φξ, φη = gφ*tξ, gφ*tη

εm = [tξ ⋅ uξ, tη ⋅ uη, (tξ ⋅ uη) + (tη ⋅ uξ)]
κb = [tη ⋅ φξ, -(tξ ⋅ φη), (tη ⋅ φη) - (tξ ⋅ φξ)]

Nξ, Nη, Nξη, Mξ, Mη, Mξη = D_mb * [εm; κb]

γξ = (n ⋅ uξ) + (φe ⋅ tη)
γη = (n ⋅ uη) - (φe ⋅ tξ)

center(s) = ScalarField(
    [repeat(sum(a; dims=1) / size(a, 1), size(a, 1), 1) for a in s.A],
    [;;], s.t, s.numElem, s.nsteps, s.type, s.model
)
Qη, Qξ = D_s * [center(γη), center(γξ)];

In [10]:
showElementResults(Nξ; name="N_xi")
showElementResults(Nη; name="N_eta")
showElementResults(Nξη; name="N_xi_eta")
showElementResults(Mξ; name="M_xi")
showElementResults(Mη; name="M_eta")
showElementResults(Mξη; name="M_xi_eta")
showElementResults(Qξ; name="Q_xi")
showElementResults(Qη; name="Q_eta");

## 7. Stress recovery in the material axes of each ply

For a point at thickness coordinate $\zeta$ inside ply $k$, its in-plane strain in material coordinates is $\mathbf T_\varepsilon(\theta_k)(\boldsymbol\varepsilon_m+\zeta\boldsymbol\kappa_b)$. Consequently, the lamina stresses are

$$
\begin{bmatrix}\sigma_{11}\\\sigma_{22}\\\tau_{12}\end{bmatrix}_k
=\mathbf Q_0\,\mathbf T_\varepsilon(\theta_k)
\left(\boldsymbol\varepsilon_m+\zeta\boldsymbol\kappa_b\right).
$$

Evaluate the stress at the center of each ply and at the two outer laminate surfaces. The stresses are evaluated separately in the material axes of each ply, without averaging between layers. The fiber stress $\sigma_{11}$ is particularly useful for comparing orientations. These plots report stress components; a composite failure criterion is not applied.

In [11]:
σply(k, ζ) = Q0 * Tε(θ[k]) * (εm + ζ*κb)

zmid = (z[1:end-1] + z[2:end]) / 2
σmid = [σply(k, zmid[k]) for k in eachindex(θ)]
σbot = σply(firstindex(θ), first(z))
σtop = σply(lastindex(θ), last(z))

for k in eachindex(θ)
    showElementResults(σmid[k][1]; name="Ply $k ($(θ[k]) deg), sigma_11")
end
showElementResults(σbot[1]; name="Bottom surface, sigma_11")
showElementResults(σtop[1]; name="Top surface, sigma_11");

## 8. Free-edge response

Integrate the nodal displacement and rotation components along `right` and divide by the edge length. The resulting mean values describe the extension, transverse displacement and rotation of the loaded edge without forcing it to remain rigid. The nonzero laminate coupling terms can produce additional twisting under the combined load.

In [12]:
u_right = [∫(U, "right", elementsToNodes(u[i])) / Ly for i in 1:3]
φ_right = [∫(Φ, "right", elementsToNodes(φ[i])) / Ly for i in 1:3]

(; displacement_mm=u_right, rotation_rad=φ_right)

(displacement_mm = [0.03024033104490937, 0.0031741902285573388, -1.047247075742908], rotation_rad = [-0.010516875062856926, 0.01049364966669308, 8.442858608274036e-6])

The saved calculation gives a mean extension of approximately 0.03024 mm and a transverse displacement of −1.04725 mm. The free-edge rotations about both the global $x$ and $y$ axes are nonzero, illustrating the bending–twisting response of the unsymmetric laminate.

The Gmsh postprocessor allows inspection of the deformed midsurface and the stress resultant and ply-stress fields. To explore the role of the ply arrangement, change `θ` and recompute the laminate stiffnesses, assembly and solution. A symmetric stacking sequence has a zero membrane–bending coupling matrix (up to numerical quadrature roundoff).

In [13]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
